# Hotel Bar Inventory Optimization
**End-to-End Solution**

### Executive Summary:
**1. Business Problem:**
Solving for stockouts (lost revenue) vs. overstocking (waste/capital). We need a data-driven way to order.

**2. Approach:**
- **Model**: Par Level System (Industry Standard). Simple Min-Max logic.
- **Assumptions**: 3-day Lead Time, 95% Service Level target.
- **Performance**: Validated via simulation to maintain >90% availability.

**3. Real-world Usage:**
Run weekly -> Print "Order Sheets" -> Staff counts -> Order placed.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# using our custom modules
from data_loader import load_and_process_data
from forecast_model import calculate_par_levels
from simulator import run_simulation

# config
FILE_PATH = "Copy of Consumption Dataset - Dataset.csv"
LEAD_TIME = 3
Z_SCORE = 1.65 # 95% confidence

sns.set_theme(style="whitegrid")


### 1. Data Prep
Loading raw pos dump, cleaning cols, and aggregating to daily consumption.


In [ ]:
df = load_and_process_data(FILE_PATH)
print(f"Loaded {len(df)} daily records")
df.head()


### 2. Quick EDA
Just checking our top movers and what the demand curve looks like.


In [ ]:
# check top volume items
top_brands = df.groupby('Brand Name')['Consumed (ml)'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_brands.values, y=top_brands.index, palette='viridis')
plt.title('Top 10 Brands by Volume')
plt.show()

# look at daily trend for the #1 item
top_item = top_brands.index[0]
subset = df[df['Brand Name'] == top_item]

plt.figure(figsize=(12, 5))
plt.plot(subset['Date'], subset['Consumed (ml)'], alpha=0.8)
plt.title(f'Daily Usage: {top_item}')
plt.show()


### 3. Calculate Par Levels
Using the standard formula: $Par = (AvgUsage 	imes LeadTime) + SafetyStock$


In [ ]:
# generating recommendations
pars = calculate_par_levels(df, lead_time_days=LEAD_TIME, service_level_z=Z_SCORE)

# let's see the high velocity items
print("Recommended Pars (Top 5):")
print(pars[['Bar Name', 'Brand Name', 'mean_daily_usage', 'recommended_par_level_ml', 'par_bottles_750ml']]
      .sort_values('mean_daily_usage', ascending=False)
      .head()
      .to_string(index=False))


### 4. Backtesting (Simulation)
Replaying history to see if these Pars actually work (target: >95% service level).


In [ ]:
sim_results = run_simulation(df, pars, lead_time_days=LEAD_TIME)

# results
avg_sl = sim_results['Service Level'].mean()
print(f"Global Service Level: {avg_sl:.2%}")

# check for any problem children (low service level)
print("\nItems needing attention (<90% SL):")
issues = sim_results[sim_results['Service Level'] < 0.9]
if not issues.empty:
    print(issues[['Bar Name', 'Brand Name', 'Service Level']].head())
else:
    print("None. All items behaving well.")
